In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Carregar a base inteira, pulando as primeiras 6 linhas
dados = pd.read_excel("/indicadores_fluxo_es_2016-2022/Dataset Projeto Dados 2016 - 2022.xlsx", skiprows=6)

# Filtrar a linha 9
dados = dados.drop(8)

# Remover as linhas 1 e 2
dados = dados.drop([0, 1])

# ---------- 

In [3]:
# Filtrar apenas os dados da Universidade de São Paulo (USP)
dados_usp = dados[dados['Nome da Instituição'] == "UNIVERSIDADE DE SÃO PAULO"]

# Remover duplicatas considerando todas as colunas especificadas
dados_usp_sem_duplicatas = dados_usp.drop_duplicates(subset=['Nome da Instituição', 'Ano de Ingresso', 'Nome do Curso de Graduação', 'Ano de Referência', 'Quantidade de Ingressantes no Curso', 'Quantidade de Permanência no Curso no ano de referência', 'Quantidade de Concluintes no Curso no ano de referência', 'Quantidade de Desistência no Curso no ano de referência'])

print(dados_usp_sem_duplicatas.shape)

In [7]:
# Criar uma cópia dos dados para evitar o aviso SettingWithCopyWarning
dados_usp_sem_duplicatas = dados_usp_sem_duplicatas.copy()

# Convertendo a coluna 'Quantidade de Ingressantes no Curso' para o tipo de dados numérico
dados_usp_sem_duplicatas['Quantidade de Ingressantes no Curso'] = pd.to_numeric(dados_usp_sem_duplicatas['Quantidade de Ingressantes no Curso'], errors='coerce')

# Convertendo as colunas para o tipo de dados numérico
colunas_numericas = ['Quantidade de Permanência no Curso no ano de referência',
                     'Quantidade de Concluintes no Curso no ano de referência',
                     'Quantidade de Desistência no Curso no ano de referência']

for coluna in colunas_numericas:
    dados_usp_sem_duplicatas[coluna] = pd.to_numeric(dados_usp_sem_duplicatas[coluna], errors='coerce')

# Selecionar apenas as colunas necessárias após remover as duplicatas de 'Ano de Ingresso'
colunas_selecionadas = ["Nome da Instituição", "Nome do Curso de Graduação", 
                        "Ano de Ingresso", "Ano de Referência", 
                        "Quantidade de Ingressantes no Curso", 
                        "Quantidade de Permanência no Curso no ano de referência", 
                        "Quantidade de Concluintes no Curso no ano de referência", 
                        "Quantidade de Desistência no Curso no ano de referência"]

dados_usp_selecionados = dados_usp_sem_duplicatas[colunas_selecionadas]

# Verificar se ainda há dados duplicados
dados_usp_selecionados.duplicated().sum()


# print(dados_usp_selecionados.shape)

dados_usp_selecionados.head(5)

,Nome da Instituição,Nome do Curso de Graduação,Ano de Ingresso,Ano de Referência,Quantidade de Ingressantes no Curso,Quantidade de Permanência no Curso no ano de referência,Quantidade de Concluintes no Curso no ano de referência,Quantidade de Desistência no Curso no ano de referência
14975,UNIVERSIDADE DE SÃO PAULO,DIREITO,2016,2016,456,450,1,5
14976,UNIVERSIDADE DE SÃO PAULO,DIREITO,2016,2017,456,443,0,7
14977,UNIVERSIDADE DE SÃO PAULO,DIREITO,2016,2018,456,433,0,10
14978,UNIVERSIDADE DE SÃO PAULO,DIREITO,2016,2019,456,415,7,11
14979,UNIVERSIDADE DE SÃO PAULO,DIREITO,2016,2020,456,143,262,10


**RANKEANDO OS CURSOS COM MAIOR QUANTIDADE DE INGRESSANTES**

In [11]:
# Remover duplicatas considerando todas as colunas relevantes, incluindo a quantidade de ingressantes
dados_usp_sem_duplicatas = dados_usp_selecionados.drop_duplicates(subset=['Nome da Instituição', 'Nome do Curso de Graduação', 'Ano de Ingresso', 'Quantidade de Ingressantes no Curso'])

# Calcular a soma dos ingressantes considerando todas as entradas únicas para cada curso
ingressantes_por_curso = dados_usp_sem_duplicatas.groupby('Nome do Curso de Graduação')['Quantidade de Ingressantes no Curso'].sum()

print(ingressantes_por_curso)

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Função para plotar o gráfico com base na seleção do curso
def plot_top_cursos(curso_selecionado):
    plt.figure(figsize=(10, 8))

    # Se "TOP 20" estiver selecionado, plotar o gráfico dos top 20 cursos
    if curso_selecionado == 'TOP 20':
        top_20_cursos.plot(kind='barh', color=cores)

        # Adicionar o número exato de ingressantes após as barras
        for i, v in enumerate(top_20_cursos):
            plt.text(v + 3, i, str(v), color='black')  # Adicionar o número após a barra

        plt.xlabel('Número de Ingressantes')
        plt.ylabel('Curso')
        plt.title('Top 20 Cursos com Mais Ingressantes (incluindo "Outros" e "Total")')
        plt.gca().invert_yaxis()  # Inverter a ordem dos cursos para o maior aparecer primeiro
        plt.legend(title="Legendas", bbox_to_anchor=(1, 1))
        plt.show()
    else:
        # Plotar o gráfico para o curso selecionado
        dados_curso_selecionado = dados_usp_sem_duplicatas[dados_usp_sem_duplicatas['Nome do Curso de Graduação'] == curso_selecionado]

        # Definir tamanho máximo da barra para cursos específicos
        max_width = max(dados_curso_selecionado['Quantidade de Ingressantes no Curso']) * 1.2

        # Ajustar a largura máxima da barra se houver apenas um curso selecionado e a largura calculada for maior que 20
        if len(dados_curso_selecionado) == 1 and max_width > 20:
            max_width = 20  # Definir um tamanho máximo de barra mais moderado

        # Plotar o gráfico para o curso selecionado
        plt.barh(dados_curso_selecionado['Nome do Curso de Graduação'], dados_curso_selecionado['Quantidade de Ingressantes no Curso'], color='blue')
        plt.xlabel('Número de Ingressantes')
        plt.ylabel('Curso')
        plt.title(f'Quantidade de Ingressantes para o Curso de {curso_selecionado}')
        plt.xlim(0, max_width)  # Definir o limite máximo do eixo x
        plt.show()

# Lista de cursos disponíveis para seleção
cursos_para_selecao = ['TOP 20'] + list(ingressantes_por_curso.index)

# Criar dropdown para selecionar o curso
dropdown_curso_selecao = widgets.Dropdown(options=cursos_para_selecao, description='Curso:')

# Exibir o dropdown e o gráfico interativo
widgets.interactive(plot_top_cursos, curso_selecionado=dropdown_curso_selecao)


Nome do Curso de Graduação
ADMINISTRAÇÃO                                                                    335
ARQUITETURA E URBANISMO                                                          204
ARTES CÊNICAS                                                                      8
ARTES CÊNICAS COM HABILITAÇÃO EM INTERPRETAÇÃO TEATRAL                             6
ARTES VISUAIS COM HABILITAÇÃO EM GRAVURA                                          10
ASTRONOMIA                                                                        20
BIBLIOTECONOMIA                                                                   38
BIBLIOTECONOMIA E CIÊNCIA DA INFORMAÇÃO                                           23
CIÊNCIA DA COMPUTAÇÃO                                                             51
CIÊNCIAS AGRÁRIAS                                                                 49
CIÊNCIAS ATUARIAIS                                                                52
CIÊNCIAS BIOLÓGICAS                   

interactive(children=(Dropdown(description='Curso:', options=('TOP 20', 'ADMINISTRAÇÃO', 'ARQUITETURA E URBANI…

<Figure size 1000x800 with 0 Axes>

**CALCULANDO A TAXA DE CONCLUINTES DE 2016 Á 2022**

In [15]:
# Filtrar os dados para incluir apenas os anos de 2016 a 2022
dados_2016_2022 = dados_usp_selecionados[(dados_usp_selecionados['Ano de Referência'] >= 2016) & (dados_usp_selecionados['Ano de Referência'] <= 2022)]

# Agrupar os dados pelo nome do curso de graduação e somar a quantidade de concluintes para cada curso em cada ano
concluintes_por_curso = dados_2016_2022.groupby(['Nome do Curso de Graduação'])['Quantidade de Concluintes no Curso no ano de referência'].sum()

# Configurar o pandas para exibir todas as linhas e colunas
pd.set_option('display.max_rows', None)

# Exibir a quantidade total de conclusões de cada curso de 2016 a 2022
print("Quantidade total de conclusões de cada curso de 2016 a 2022:")
print(concluintes_por_curso)


Quantidade total de conclusões de cada curso de 2016 a 2022:
Nome do Curso de Graduação
ADMINISTRAÇÃO                                                                    267
ARQUITETURA E URBANISMO                                                          142
ARTES CÊNICAS                                                                      6
ARTES CÊNICAS COM HABILITAÇÃO EM INTERPRETAÇÃO TEATRAL                             5
ARTES VISUAIS COM HABILITAÇÃO EM GRAVURA                                           7
ASTRONOMIA                                                                         9
BIBLIOTECONOMIA                                                                   16
BIBLIOTECONOMIA E CIÊNCIA DA INFORMAÇÃO                                           16
CIÊNCIA DA COMPUTAÇÃO                                                             35
CIÊNCIAS AGRÁRIAS                                                                 24
CIÊNCIAS ATUARIAIS                                            

In [17]:
# Remover duplicatas dos dados de entrada, mantendo apenas a primeira ocorrência de cada conjunto de duplicatas com a mesma quantidade de ingressantes
dados_sem_duplicatas = dados_2016_2022.drop_duplicates(subset=['Nome do Curso de Graduação', 'Quantidade de Ingressantes no Curso', 'Ano de Ingresso'])

# Calcular o total de ingressantes em cada curso de graduação
total_ingressantes_por_curso = dados_sem_duplicatas.groupby('Nome do Curso de Graduação')['Quantidade de Ingressantes no Curso'].sum()

# Calcular o total de concluintes em cada curso de graduação
total_concluintes_por_curso = dados_2016_2022.groupby('Nome do Curso de Graduação')['Quantidade de Concluintes no Curso no ano de referência'].sum()

# Exibir o total de ingressantes em cada curso de graduação
print("Total de ingressantes em cada curso de graduação:")
print(total_ingressantes_por_curso)

# Exibir o total de concluintes em cada curso de graduação
print("\nTotal de concluintes em cada curso de graduação:")
print(total_concluintes_por_curso)

# Calcular a porcentagem de concluintes em relação ao total de ingressantes para cada curso de graduação
porcentagem_concluintes_ingressantes = (total_concluintes_por_curso / total_ingressantes_por_curso) * 100

# Configurar o pandas para exibir todas as linhas e colunas
pd.set_option('display.max_rows', None)

# Exibir a porcentagem de concluintes em relação ao total de ingressantes para cada curso de graduação
print("\nPorcentagem de concluintes em relação ao total de ingressantes para cada curso de graduação:")
print(porcentagem_concluintes_ingressantes)



Total de ingressantes em cada curso de graduação:
Nome do Curso de Graduação
ADMINISTRAÇÃO                                                                    335
ARQUITETURA E URBANISMO                                                          204
ARTES CÊNICAS                                                                      8
ARTES CÊNICAS COM HABILITAÇÃO EM INTERPRETAÇÃO TEATRAL                             6
ARTES VISUAIS COM HABILITAÇÃO EM GRAVURA                                          10
ASTRONOMIA                                                                        20
BIBLIOTECONOMIA                                                                   38
BIBLIOTECONOMIA E CIÊNCIA DA INFORMAÇÃO                                           23
CIÊNCIA DA COMPUTAÇÃO                                                             51
CIÊNCIAS AGRÁRIAS                                                                 49
CIÊNCIAS ATUARIAIS                                                       

In [18]:
# Calcular as taxas de permanência, desistência e conclusão por curso e ano de referência
taxas = dados_usp_selecionados.groupby(['Nome do Curso de Graduação', 'Ano de Referência']).agg({
    'Quantidade de Permanência no Curso no ano de referência': 'sum',
    'Quantidade de Desistência no Curso no ano de referência': 'sum',
    'Quantidade de Concluintes no Curso no ano de referência': 'sum',
    'Quantidade de Ingressantes no Curso': 'sum'
}).reset_index()

taxas['Taxa_Permanencia'] = (taxas['Quantidade de Permanência no Curso no ano de referência'] / taxas['Quantidade de Ingressantes no Curso']) * 100
taxas['Taxa_Desistencia'] = (taxas['Quantidade de Desistência no Curso no ano de referência'] / taxas['Quantidade de Ingressantes no Curso']) * 100
taxas['Taxa_Conclusao'] = (taxas['Quantidade de Concluintes no Curso no ano de referência'] / taxas['Quantidade de Ingressantes no Curso']) * 100
taxas['Taxa_Retencao'] = taxas['Taxa_Permanencia'] + taxas['Taxa_Conclusao']

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Função para atualizar os gráficos com base na seleção do curso
def update_plot(curso):
    plt.figure(figsize=(15, 5))

    # Gráfico de Taxa de Permanência
    plt.subplot(1, 3, 1)
    data_curso = taxas[taxas['Nome do Curso de Graduação'] == curso]
    plt.bar(data_curso['Ano de Referência'], data_curso['Taxa_Permanencia'], color='blue')
    plt.xlabel('Ano de Referência')
    plt.ylabel('Taxa de Permanência (%)')
    plt.title(f'Taxa de Permanência para o Curso de {curso}')

    # Gráfico de Taxa de Conclusão
    plt.subplot(1, 3, 2)
    plt.bar(data_curso['Ano de Referência'], data_curso['Taxa_Conclusao'], color='green')
    plt.xlabel('Ano de Referência')
    plt.ylabel('Taxa de Conclusão (%)')
    plt.title(f'Taxa de Conclusão para o Curso de {curso}')

    # Gráfico de Taxa de Desistência
    plt.subplot(1, 3, 3)
    plt.bar(data_curso['Ano de Referência'], data_curso['Taxa_Desistencia'], color='red')
    plt.xlabel('Ano de Referência')
    plt.ylabel('Taxa de Desistência (%)')
    plt.title(f'Taxa de Desistência para o Curso de {curso}')

    plt.tight_layout()
    plt.show()

# Lista de cursos disponíveis
cursos_disponiveis = taxas['Nome do Curso de Graduação'].unique().tolist()

# Criar dropdown para selecionar o curso
dropdown_curso = widgets.Dropdown(options=cursos_disponiveis, description='Curso:')

# Atualizar os gráficos quando uma nova seleção de curso for feita
widgets.interactive(update_plot, curso=dropdown_curso)

##




interactive(children=(Dropdown(description='Curso:', options=('ADMINISTRAÇÃO', 'ARQUITETURA E URBANISMO', 'ART…